# Phase III: Text Summarization Inference Pipeline

Welcome to the inference phase of our Text Summarization project! In this notebook, we will set up the "brain" of our system—a pre-trained Transformer model—and use it to generate concise summaries of long articles.

### What is an Inference Pipeline?
An **inference pipeline** is a sequence of steps that takes raw data (like a news article), processes it so a machine can understand it (tokenization), feeds it into a specialized AI model, and then converts the model's complex mathematical output back into human-readable text (a summary).

### Key Technologies Used:
- **Hugging Face Transformers**: A library providing state-of-the-art AI models.
- **BART (Bidirectional and Auto-Regressive Transformers)**: Our chosen model, specifically designed for summarizing text.
- **PyYaml**: To load our project settings from a configuration file.

## 1. Setup and Project Configuration

Before we start, we need to load our settings from `config.yaml`. This file tells us which model to use and what the maximum lengths for our articles and summaries should be.

In [1]:
import yaml
import os
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load configuration
config_path = os.path.join("..", "config.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

model_name = config['model']['name']
max_input = config['model']['max_input_length']
max_output = config['model']['max_output_length']

print(f"Using model: {model_name}")
print(f"Max input length: {max_input} tokens")
print(f"Max output length: {max_output} tokens")

c:\Users\My Device\Desktop\Text Summarization Using Pre-trained Models\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using model: facebook/bart-base
Max input length: 1024 tokens
Max output length: 128 tokens


## 2. Loading the Model and Tokenizer

AI models don't read text like we do; they process numbers. 
- The **Tokenizer** breaks text into smaller chunks called "tokens" and maps them to numbers.
- The **Model** (BART) takes those numbers and predicts the next most likely words for our summary.

In [2]:
print("Loading tokenizer and model... This may take a moment depending on your internet connection.")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


print(f"Model loaded successfully on: {device}")

Loading tokenizer and model... This may take a moment depending on your internet connection.


c:\Users\My Device\Desktop\Text Summarization Using Pre-trained Models\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model loaded successfully on: cpu


## 3. Creating the Summarization Pipeline

Hugging Face provides a "pipeline" object that simplifies the entire process. It handles tokenization, model prediction, and decoding automatically.

In [3]:
import sys
import transformers
print(f"Python Executable: {sys.executable}")
print(f"Transformers Version: {transformers.__version__}")

from transformers.pipelines import PIPELINE_REGISTRY
print(f"Summarization Task Available: {'summarization' in PIPELINE_REGISTRY.get_supported_tasks()}")


Python Executable: c:\Users\My Device\Desktop\Text Summarization Using Pre-trained Models\.venv\Scripts\python.exe
Transformers Version: 4.38.0
Summarization Task Available: True


## 4. Testing the System (Example Inference)

Now, let's feed a long news snippet into our system and see if it can capture the main points.

In [5]:
import transformers
print(f"Transformers Version: {transformers.__version__}")
from transformers.pipelines import PIPELINE_REGISTRY
print(f"Summarization Task Ready: {'summarization' in PIPELINE_REGISTRY.get_supported_tasks()}")

# Check if the variables exist in this session
print(f"Model variable defined: {'model' in locals()}")
print(f"Summarizer variable defined: {'summarizer' in locals()}")


Transformers Version: 4.38.0
Summarization Task Ready: True
Model variable defined: True
Summarizer variable defined: False


In [8]:
# The "Deep" summarization logic using manual Beam Search
inputs = tokenizer(sample_text, return_tensors="pt", max_length=1024, truncation=True).to(model.device)

summary_ids = model.generate(
    inputs["input_ids"],
    num_beams=4,
    min_length=30,
    max_length=128,
    no_repeat_ngram_size=3,
    early_stopping=True
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(summary)


The Hubble Space Telescope has captured a stunning new image of a distant galaxy cluster, Âthe largest in the universe,revealing thousands of stars and planetary systems in unprecedented detail. Scientists at NASA say Âthis discovery could help us understand the early formation of the universe. The cluster, located Âbillions of light-years away, shows signs of gravitational lensing, where the intense gravity of  the cluster bends the light of even more distant objects behind it. This effect acts like a cosmic Âmagnifying glass reaching back to the dawn of time. Â


### Conclusion
We now have a working system that can take any text and shorten it while keeping the core meaning! In the next phase, we will evaluate how "good" these summaries are compared to human-written ones using a metric called ROUGE.